# Data 400 Big Project - Data Concatenation
## Will & Kevin

In [3]:
import pandas as pd
import numpy as np
import dropbox
import os
import io
from dropbox.exceptions import ApiError
from scipy import stats

In [4]:
# Import all .csv files from Dropbox and concatenate into a single DataFrame - save this to repo
# Save dropbox credentials
DROPBOX_APP_KEY = "9gghuhtkyrtyn2u"
DROPBOX_APP_SECRET = "tqtvwbr15gbwseg"
DROPBOX_REFRESH_TOKEN = "LpMKV__by1wAAAAAAAAAAYn_oi2fIW0SBBqF0n6gap1WavDnJGFRMcVWQol7X5Vy"

# Create Dropbox client
dbx = dropbox.Dropbox(
    app_key=DROPBOX_APP_KEY,
    app_secret=DROPBOX_APP_SECRET,
    oauth2_refresh_token=DROPBOX_REFRESH_TOKEN
)

print("Reading files from Dropbox...")

# List all files
result = dbx.files_list_folder('/flight_data')

dfs = []
for entry in result.entries:
    if entry.name.endswith('.csv'):
        print(f"Reading: {entry.name}")
        
        # Download file content
        metadata, response = dbx.files_download(entry.path_lower)
        
        # Read directly into pandas
        df = pd.read_csv(io.BytesIO(response.content))
        dfs.append(df)

# Combine
combined_df = pd.concat(dfs, ignore_index=True)

# Process
combined_df['route'] = combined_df['origin'] + ' → ' + combined_df['destination']
combined_df['time_collected'] = pd.to_datetime(combined_df['time_collected'])
combined_df['departure_date'] = pd.to_datetime(combined_df['departure_date'])

# Save locally
combined_df.to_csv('combined_flight_data.csv', index=False)

# Load combined data
df = pd.read_csv("combined_flight_data.csv")

# Convert to datetime with error handling
try:
    if df['departure_date'].dtype != 'datetime64[ns]':
        df['departure_date'] = pd.to_datetime(df['departure_date'], errors='coerce')
except Exception as e:
    print(f"Error with departure_date: {e}")
    # Try alternative parsing
    df['departure_date'] = pd.to_datetime(df['departure_date'].astype(str), errors='coerce')

try:
    if df['departure_time'].dtype != 'datetime64[ns]':
        df['departure_time'] = pd.to_datetime(df['departure_time'], errors='coerce')
except Exception as e:
    print(f"Error with departure_time: {e}")
    df['departure_time'] = pd.to_datetime(df['departure_time'].astype(str), errors='coerce')

try:
    if df['time_collected'].dtype != 'datetime64[ns]':
        df['time_collected'] = pd.to_datetime(df['time_collected'], errors='coerce')
except Exception as e:
    print(f"Error with time_collected: {e}")
    df['time_collected'] = pd.to_datetime(df['time_collected'].astype(str), errors='coerce')

# Round departure time to nearest 15 minutes (flights might vary by a few minutes)
df['departure_time_rounded'] = df['departure_time'].dt.floor('15min')

# Create flight ID for each unique flight
df['flight_id'] = (
    df['origin'].astype(str) + '_' + 
    df['destination'].astype(str) + '_' + 
    df['departure_date'].dt.strftime('%Y%m%d') + '_' + 
    df['departure_time_rounded'].dt.strftime('%H%M') + '_' + 
    df['airline'].astype(str)
)
print(f"Number of Observations: {len(df)}")
df.head()

Reading files from Dropbox...
Reading: flight_data_20251110_210135.csv
Reading: flight_data_20251111_031726.csv
Reading: flight_data_20251111_130430.csv
Reading: flight_data_20251112_021940.csv
Reading: flight_data_20251112_130852.csv
Reading: flight_data_20251113_022155.csv
Reading: flight_data_20251113_130822.csv
Reading: flight_data_20251114_022042.csv
Reading: flight_data_20251114_130600.csv
Reading: flight_data_20251115_021721.csv
Reading: flight_data_20251115_125944.csv
Reading: flight_data_20251116_022726.csv
Reading: flight_data_20251116_125744.csv
Reading: flight_data_20251117_022452.csv
Reading: flight_data_20251117_132325.csv
Reading: flight_data_20251118_025158.csv
Reading: flight_data_20251118_130553.csv
Reading: flight_data_20251119_021933.csv
Reading: flight_data_20251119_130946.csv
Reading: flight_data_20251120_021830.csv
Reading: flight_data_20251120_130617.csv
Reading: flight_data_20251121_022030.csv
Reading: flight_data_20251121_130024.csv
Reading: flight_data_202511

,time_collected,origin,destination,departure_date,days_until_departure,price,currency,airline,number_of_stops,departure_time,arrival_time,total_duration,aircraft_type,cabin_class,bookable_seats,route,departure_time_rounded,flight_id
0,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,144.27,EUR,F9,1,2025-11-11 06:25:00,2025-11-11T12:25:00,PT9H,"32Q,32Q",ECONOMY,4,JFK → LAX,2025-11-11 06:15:00,JFK_LAX_20251111_0615_F9
1,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,149.13,EUR,F9,1,2025-11-11 07:59:00,2025-11-11T19:53:00,PT14H54M,"32Q,32Q",ECONOMY,4,JFK → LAX,2025-11-11 07:45:00,JFK_LAX_20251111_0745_F9
2,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,149.13,EUR,F9,1,2025-11-11 07:59:00,2025-11-11T20:42:00,PT15H43M,"32Q,32N",ECONOMY,4,JFK → LAX,2025-11-11 07:45:00,JFK_LAX_20251111_0745_F9
3,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,210.02,EUR,F9,0,2025-11-11 11:29:00,2025-11-11T14:32:00,PT6H3M,32Q,ECONOMY,4,JFK → LAX,2025-11-11 11:15:00,JFK_LAX_20251111_1115_F9
4,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,256.07,EUR,AS,1,2025-11-11 07:00:00,2025-11-11T15:17:00,PT11H17M,"73H,73J",ECONOMY,7,JFK → LAX,2025-11-11 07:00:00,JFK_LAX_20251111_0700_AS


In [6]:
df.to_csv('../data/final_flight_data.csv', index=False)